In [1]:
from typing import List

import torch
import pytorch_lightning as pl
from torch import Tensor, nn
from torch.nn import functional as F
from torchmetrics import JaccardIndex, F1Score
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.models.segmentation.deeplabv3 import ASPP
from pytorch_lightning.loggers import TensorBoardLogger
import torch
import torch.nn as nn
import torch.nn.functional as F
from kornia import losses

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchmetrics.classification import BinaryAccuracy
from pytorch_lightning.callbacks import RichProgressBar
import numpy as np
import glob
import os
import rasterio
import re
from pytorch_lightning.callbacks import ModelCheckpoint

import tifffile as tiff
import matplotlib.pyplot as plt

from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
def _reshape_maks(inputs, targets, ignore_index: int):
    # do everything using a 1d array
    if inputs.dim() > 2:
        # N,C,H,W => N,C,H*W
        inputs = inputs.view(inputs.size(0), inputs.size(1), -1)
        # N,C,H*W => N,H*W,C
        inputs = inputs.transpose(1, 2)
        # N,H*W,C => N*H*W,C
        inputs = inputs.contiguous().view(-1, inputs.size(2))    

    targets = targets.view(-1, 1)

    # drop ignored_index
    mask = targets==ignore_index
    targets = targets[~mask.ravel(),:].ravel()
    inputs = inputs[~mask.ravel(),:]

    return inputs, targets


class FocalLossMod(nn.Module):
    
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean', ignore_index=-100):
        super(FocalLossMod, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.ignore_index = ignore_index

    def forward(self, inputs_in, targets_in):

        # clone tensors to not disturb tensors that were provided
        inputs = torch.clone(inputs_in)
        targets = torch.clone(targets_in)

        inputs, targets = _reshape_maks(inputs, targets, self.ignore_index)

        return losses.focal_loss(inputs, targets, self.alpha, self.gamma, self.reduction)

class DiceLossMod(nn.Module):
    """Dice Loss modified from kornia to be able to handle "ignore_index"
    https://kornia.readthedocs.io/en/latest/_modules/kornia/losses/dice.html#DiceLoss
    """
    
    def __init__(self, ignore_index=-100):
        super(DiceLossMod, self).__init__()
        self.ignore_index = ignore_index

    def forward(self, inputs_in, targets_in):

        if not isinstance(inputs_in, torch.Tensor):
            raise TypeError(f"Input type is not a torch.Tensor. Got {type(inputs_in)}")

        if not inputs_in.device == targets_in.device:
            raise ValueError(f"input and target must be in the same device. Got: {inputs_in.device} and {targets_in.device}")

        eps: float = 1e-8
        num_classes = inputs_in.shape[1]

        # clone tensors to not disturb tensors that were provided
        inputs = torch.clone(inputs_in)
        targets = torch.clone(targets_in)

        inputs, targets = _reshape_maks(inputs, targets, self.ignore_index)

        # compute softmax over the classes axis
        inputs_soft: torch.Tensor = F.softmax(inputs, dim=1)

        # create the labels one hot tensor
        targets_one_hot: torch.Tensor = F.one_hot(targets, num_classes=num_classes)

        # compute the actual dice score
        intersection = torch.sum(inputs_soft * targets_one_hot)
        cardinality = torch.sum(inputs_soft + targets_one_hot)

        dice_score = 2.0 * intersection / (cardinality + eps)

        return torch.mean(-dice_score + 1.0)


In [3]:
###############################################
###############################################
class ResNetASPP(pl.LightningModule):
    def __init__(self, *args, **kwargs):
        super().__init__()

        self.save_hyperparameters()

        if kwargs['pretrained'] == True:
            weights=ResNet18_Weights.IMAGENET1K_V1
        else:
            weights=None

        return_layers = {"layer2": "out"}
        self.encoder = resnet18(weights=weights)
        self.encoder = IntermediateLayerGetter(self.encoder, return_layers=return_layers)

        if self.hparams.frozen_start:
            for param in self.encoder.parameters():
                param.requires_grad = False
    
        # create the head:
        self.decoder = nn.Sequential(ASPP(in_channels=128, atrous_rates = [12, 24, 36], out_channels = 128),
                                        nn.Conv2d(128, 128, 3, padding=1, bias=False),
                                        nn.BatchNorm2d(128),
                                        nn.ReLU(),
                                        nn.Conv2d(128, self.hparams['num_classes'], 1)
                                        )

        # ASPP has one global average pooling that messes things up 
        # in case we want to change the input size (full raster prediction)
        avgpool_replacer = nn.AvgPool2d(8,8)
        if isinstance(self.decoder[0].convs[-1][0], nn.AdaptiveAvgPool2d):
            self.decoder[0].convs[-1][0] = avgpool_replacer
        else:
            print('Check the model! Is there an AdaptiveAvgPool2d somewhere?')

        self.jaccard = JaccardIndex(num_classes=self.hparams.num_classes+1, 
                                    average='macro',
                                    task="binary")
        self.f1 = F1Score(num_classes=self.hparams.num_classes+1, 
                                    average='macro',
                                    task="binary")
        
        self.acc = BinaryAccuracy()

        # loss
        if self.hparams.loss == 'cross_entropy':
            self.loss_fn = nn.CrossEntropyLoss()
        elif self.hparams.loss == 'focal':
            self.loss_fn = FocalLossMod(gamma=self.hparams.gamma, 
                                        alpha=self.hparams.alpha, 
                                        reduction='mean')

        elif self.hparams.loss == 'dice':
            self.loss_fn = DiceLossMod()

        
    def forward(self, x:Tensor) -> Tensor:
        
        input_shape = x.shape[-2:]

        features = self.encoder(x)['out']

        logits = self.decoder(features)         
        logits = F.interpolate(logits, size=input_shape, mode="bilinear", align_corners=False)

        return logits
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 
                                                                    min_lr=1e-8,
                                                                    patience=self.hparams.reduce_lr_patience, 
                                                                    verbose=True)         
        return optimizer 
    
    def training_step(self, batch, batch_idx):

        x, y = batch
        # torchmetrics F1 does not allow arbitrary "ignore_index"
        y_pred = self.forward(x)
        loss = self.loss_fn(y_pred, y)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)

        ###############################################
        # metrics
        ###############################################
        # IoU/Jaccard index
        iou = self.jaccard(torch.argmax(y_pred, axis=1), y)
        self.log('train_IoU', iou, on_epoch=True, prog_bar=True)

        # F1 
        f1 = self.f1(torch.argmax(y_pred, axis=1).ravel(), y.ravel())
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)

        acc = self.acc(torch.argmax(y_pred, axis=1), y)
        self.log('train_acc', acc, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch

        y_pred = self.forward(x)
        loss = self.loss_fn(y_pred, y)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)

        ###############################################
        # metrics
        ###############################################
        # IoU/Jaccard index
        iou = self.jaccard(torch.argmax(y_pred, axis=1), y)
        self.log('val_IoU', iou, on_epoch=True, prog_bar=True)

        # F1
        f1 = self.f1(torch.argmax(y_pred, axis=1).ravel(), y.ravel())
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)
        
        acc = self.acc(torch.argmax(y_pred, axis=1), y)
        self.log('val_acc', acc, on_epoch=True, prog_bar=True)

        return loss

    def on_train_epoch_end(self) -> None:
    # ReduceLROnPlateau expects val_loss
        sch = self.scheduler
        if isinstance(sch, torch.optim.lr_scheduler.ReduceLROnPlateau):
            val_loss = self.trainer.callback_metrics.get("val_loss")
            if val_loss is not None:
                sch.step(val_loss)

                # Unfreeze encoder if frozen_start and learning rate decreased
                if self.hparams.frozen_start and (sch.optimizer.param_groups[0]['lr'] < self.hparams.lr):
                    for param in self.encoder.parameters():
                        param.requires_grad = True
                  

In [4]:
def load_data_new(input_dir, mask_dir):

    # Ensure the directories exist
    if not os.path.exists(input_dir):
        print(f"Input directory {input_dir} does not exist.")
        return None, None
    
    if not os.path.exists(mask_dir):
        print(f"Mask directory {mask_dir} does not exist.")
        return None, None

    # Search for .tif files in the directories
    input_files = glob.glob(os.path.join(input_dir, '*.tif'))
    mask_files = glob.glob(os.path.join(mask_dir, '*.tif'))

    # print("Found input files:", input_files)
    # print("Found mask files:", mask_files)

    if not input_files:
        print(f"No input files found in {input_dir}.")
        return None, None

    if not mask_files:
        print(f"No mask files found in {mask_dir}.")
        return None, None

    images = []
    masks = []

    for mask_file in mask_files:
        # Extract the number i from the mask filename
        match = re.search(r'NDWI_Mask_(\d+)_resized_corrupt.tif', os.path.basename(mask_file))
        if match:
            i = match.group(1)
            input_file = os.path.join(input_dir, f'{i}.tif')
            # print("Input File : ",input_file)
            # Check if the corresponding input file exists
            if os.path.exists(input_file):
                # Read input file
                with rasterio.open(input_file) as src:

                    # print("Width x Height:", src.width, "x", src.height)
                    # print("Number of bands (channels):", src.count)
                    # print("CRS:", src.crs)           # Optional: coordinate reference system
                    # print("Bounds:", src.bounds)     # Optional: extent
                    # print(src.meta)          # General info
                    # print(src.descriptions)
                    img = src.read(1)  # Read the first band assuming it's a single-band image
                    images.append(img)

                # Read mask file
                with rasterio.open(mask_file) as src:
                    
                    # print("Width x Height:", src.width, "x", src.height)
                    # print("Number of bands (channels):", src.count)
                    # print("CRS:", src.crs)           # Optional: coordinate reference system
                    # print("Bounds:", src.bounds) 
                    # print(src.meta)          # General info
                    # print(src.descriptions)
                    msk = src.read(1)  # Read the first band assuming it's a single-band image
                    masks.append(msk)
            else:
                print(f"Corresponding input file {input_file} for mask {mask_file} not found.")

    if not images or not masks:
        print("No matching pairs of images and masks found.")
        return None, None

    return np.array(images), np.array(masks)


In [5]:
class NDWIDataset(torch.utils.data.Dataset):
    def __init__(self, images=None, masks=None,
                 input_dir=None, mask_dir=None):

        if images is None and masks is None:
            # normal loading
            self.images, self.masks = load_data_new(input_dir, mask_dir)
        else:
            # use already-split arrays
            self.images = images
            self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32)   # (H, W)
        mask = self.masks[idx].astype(np.int64)     # (H, W)

        img = np.repeat(img[None, :, :], 3, axis=0) # (3, H, W)

        img = torch.tensor(img, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        return img, mask

In [6]:
def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)


def iou(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


def save_image(image, filepath):
    image = (image * 255).astype(np.uint8)
    tiff.imwrite(filepath, image)


def predict1(
    model,
    mask_dir,
    image_dir,
    output_dir,
    device="cuda",
    number=-1
):
    dice_scores = []
    iou_scores = []
    precisions = []
    recalls = []
    f1_scores = []

    os.makedirs(output_dir, exist_ok=True)

    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('_resized.tif')]
    print(f"Found {len(mask_files)} mask files")

    model = model.to(device)
    model.eval()  # deterministic prediction (same as TF predict)

    i = 0
    for mask_file in mask_files:

        # ---------- filename logic (unchanged) ----------
        i_str = mask_file.split('_')[2]
        image_file = f"{i_str}.tif"

        mask_path = os.path.join(mask_dir, mask_file)
        image_path = os.path.join(image_dir, image_file)

        # ---------- load image & mask ----------
        arbitrary_img = tiff.imread(image_path)
        arbitrary_mask = tiff.imread(mask_path)
        
        # Ensure H x W x 1
        if len(arbitrary_img.shape) == 2:
            arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)
        elif arbitrary_img.shape[0] == 2:
            arbitrary_img = np.expand_dims(arbitrary_img[0], axis=-1)

        # ---------- to torch tensor ----------
        x = (
            torch.from_numpy(arbitrary_img)
            .permute(2, 0, 1)   # HWC → CHW
            .unsqueeze(0)      # batch dim
            .float()
            .to(device)
        )
        
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)   # (1, 3, H, W)

        # ---------- model prediction ----------
        with torch.no_grad():
            logits = model(x)                 # (1, C, H, W)
            predicted_mask = torch.argmax(logits, dim=1)  # (1, H, W)
            predicted_mask_thresh = predicted_mask.squeeze(0).cpu().numpy()

        save_image(predicted_mask_thresh,
                   f"./{output_dir}/{i_str}.tif")

        # ---------- metrics (UNCHANGED) ----------
        dice = dice_coefficient(arbitrary_mask, predicted_mask_thresh)
        iou_score = iou(arbitrary_mask, predicted_mask_thresh)

        precision = precision_score(
            arbitrary_mask.flatten(),
            predicted_mask_thresh.flatten(),
            zero_division=0
        )

        recall = recall_score(
            arbitrary_mask.flatten(),
            predicted_mask_thresh.flatten(),
            zero_division=0
        )

        f1 = f1_score(
            arbitrary_mask.flatten(),
            predicted_mask_thresh.flatten(),
            zero_division=0
        )

        dice_scores.append(dice)
        iou_scores.append(iou_score)
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

        i += 1
        if i == number:
            break

    # ---------- aggregate metrics ----------
    mean_dice = np.mean(dice_scores)
    mean_iou = np.mean(iou_scores)
    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)
    mean_f1 = np.mean(f1_scores)

    print(f"Mean Dice Coefficient: {mean_dice}")
    print(f"Mean IoU: {mean_iou}")
    print(f"Mean Precision: {mean_precision}")
    print(f"Mean Recall: {mean_recall}")
    print(f"Mean F1 Score: {mean_f1}")

    return mean_dice, mean_iou, mean_precision, mean_recall, mean_f1


In [7]:
corruptions = [2,8,15]
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="{epoch}-{val_loss:.4f}"
)

configs = ['erosion', 'mixed', 'dilation']
metrics = ['Entropy']
input_dir = './data_new/'
train = False

for corruption in corruptions:
    for config in configs:
    
        mask_dir = f'./GEE_Masks/GEE_resized/train_gee/train_{corruption}_gee_{config}'
        # Load data
        images, masks = load_data_new(input_dir, mask_dir)

        # Define input shape and number of classes
        input_shape = images.shape
        
        # Split
        train_imgs, val_imgs, train_masks, val_masks = train_test_split(
            images,
            masks,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )
        
        train_dataset = NDWIDataset(images=train_imgs, masks=train_masks)
        val_dataset = NDWIDataset(images=val_imgs, masks=val_masks)

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
                
        model_name = f'ResNetASPP_{config}_{corruption}'
        print(model_name)
        
        model = ResNetASPP(
            pretrained=True,
            frozen_start=False,
            num_classes=2,
            lr=1e-4,
            reduce_lr_patience=3,
            loss="cross_entropy",
            gamma=2.0,
            alpha=0.25
        )
        
        if train:

            logger = TensorBoardLogger("logs", name="resnet_aspp_ndwi")

            trainer = pl.Trainer(
                max_epochs=30,
                accelerator="gpu",
                devices=1,
                precision=16,
                log_every_n_steps=20,
                logger=logger,
                callbacks=[checkpoint_cb]
            )

            trainer.fit(model, train_loader, val_loader)
            trainer.save_checkpoint(f"{model_name}.ckpt")
            
        
        for metric in metrics:
            
            model = ResNetASPP.load_from_checkpoint(
                f"{model_name}.ckpt",
                map_location="cuda"  # or "cpu"
            )

            mask_dir = './GEE_Masks/GEE_resized/test_gee/'
            output_dir = f'./GEE_Output/ResNetASPP/{config}/{corruption}/{metric}/'
            os.makedirs(output_dir,exist_ok=True)
            
            if metric == "Entropy":

                predict1(
                    model,
                    mask_dir,
                    input_dir, 
                    output_dir    
                )

c:\Users\ADMIN\anaconda3\envs\torch231\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


ResNetASPP_erosion_2
Found 253 mask files
Mean Dice Coefficient: 0.7533010445218857
Mean IoU: 0.6887222706807229
Mean Precision: 0.751412301576926
Mean Recall: 0.6561414154908739
Mean F1 Score: 0.6729612300043363


c:\Users\ADMIN\anaconda3\envs\torch231\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


ResNetASPP_mixed_2
Found 253 mask files
Mean Dice Coefficient: 0.7607505867606541
Mean IoU: 0.6948227614764377
Mean Precision: 0.7111917797234281
Mean Recall: 0.724605057505804
Mean F1 Score: 0.6921849553252937


c:\Users\ADMIN\anaconda3\envs\torch231\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


...

ResNetASPP_erosion_15
Found 253 mask files
Mean Dice Coefficient: 0.3976217641497726
Mean IoU: 0.3255054979917805
Mean Precision: 0.5773408394203595
Mean Recall: 0.2554030604122973
Mean F1 Score: 0.3132817079953934


c:\Users\ADMIN\anaconda3\envs\torch231\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


ResNetASPP_mixed_15
Found 253 mask files
Mean Dice Coefficient: 0.6833907454378059
Mean IoU: 0.5967365888580937
Mean Precision: 0.654460328397657
Mean Recall: 0.6436936839744757
Mean F1 Score: 0.6241812626090665


c:\Users\ADMIN\anaconda3\envs\torch231\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


ResNetASPP_dilation_15
Found 253 mask files
Mean Dice Coefficient: 0.5169948770729103
Mean IoU: 0.4022321588962169
Mean Precision: 0.411162826141874
Mean Recall: 0.814163354164758
Mean F1 Score: 0.5169353441966787


In [8]:
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="{epoch}-{val_loss:.4f}"
)

metrics = ['Entropy']
input_dir = './data_new_gaussian/'
train = False
    
mask_dir = f'./GEE_Masks/GEE_resized/train_gee/train_gee'
# Load data
images, masks = load_data_new(input_dir, mask_dir)

# Define input shape and number of classes
input_shape = images.shape

# Split
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    images,
    masks,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_dataset = NDWIDataset(images=train_imgs, masks=train_masks)
val_dataset = NDWIDataset(images=val_imgs, masks=val_masks)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
        
model_name = f'ResNetASPP_Gaussian'
print(model_name)

model = ResNetASPP(
    pretrained=True,
    frozen_start=False,
    num_classes=2,
    lr=1e-4,
    reduce_lr_patience=3,
    loss="cross_entropy",
    gamma=2.0,
    alpha=0.25
)

if train:

    logger = TensorBoardLogger("logs", name="resnet_aspp_ndwi")

    trainer = pl.Trainer(
        max_epochs=30,
        accelerator="gpu",
        devices=1,
        precision=16,
        log_every_n_steps=20,
        logger=logger,
        callbacks=[checkpoint_cb]
    )

    trainer.fit(model, train_loader, val_loader)
    trainer.save_checkpoint(f"{model_name}.ckpt")
            
for metric in metrics:

    mask_dir = './GEE_Masks/GEE_resized/test_gee/'
    output_dir = f'./GEE_Output/ResNetASPP/Gaussian/{metric}/'
    os.makedirs(output_dir,exist_ok=True)
    
    model = ResNetASPP.load_from_checkpoint(
        f"{model_name}.ckpt",
        map_location="cuda"  # or "cpu"
    )

    if metric == "Entropy":
        
        predict1(
            model,
            mask_dir,
            input_dir, 
            output_dir    
        )
                  
                

c:\Users\ADMIN\anaconda3\envs\torch231\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


ResNetASPP_Gaussian
Found 253 mask files
Mean Dice Coefficient: 0.7697935411318821
Mean IoU: 0.7094049204939173
Mean Precision: 0.7387142563565295
Mean Recall: 0.6974289285196332
Mean F1 Score: 0.6932047048914778
